# 02_ex_<agreement>_<topic> — exploration and approval drafting

Use this notebook to profile source data, draft AI-assisted suggestions, and document human-reviewed decisions before anything is enforced in pipeline execution.

**You edit**
- Agreement/topic metadata
- Source target details
- Draft contract, DQ, and classification decisions

**This notebook produces**
- Profiling evidence
- Candidate DQ/classification suggestions
- Human-reviewed contract draft and approved decisions

**AI advisory boundary**
- AI suggestions are advisory and must be human-reviewed before promotion.


## Segment 1: Load shared config and imports

What this does: loads `00_env_config` and imports exploration helpers.

Functions used: `load_fabric_config`, `build_runtime_context`, `generate_metadata_profile`, `generate_dq_rule_candidates_with_fabric_ai`.

Output: configured runtime context and helper availability.


In [ ]:
%run 00_env_config


In [ ]:
import json
from pathlib import Path
from fabricops_kit.environment_config import bootstrap_fabric_env, get_path, load_fabric_config
from fabricops_kit.fabric_input_output import lakehouse_table_read, warehouse_read, seed_minimal_sample_source_table
from fabricops_kit.runtime_context import assert_notebook_name_valid, build_runtime_context, validate_notebook_name
from fabricops_kit.data_profiling import generate_metadata_profile, profile_dataframe_to_metadata
from fabricops_kit.data_quality import suggest_dq_rules_prompt
from fabricops_kit.data_contracts import normalize_contract_dict, validate_contract_dict, write_contract_to_lakehouse


## Agreement and approved usage


In [ ]:
USE_SAMPLE_DATA = True
DATA_AGREEMENT = "sample_minimal_agreement" if USE_SAMPLE_DATA else "TODO_replace_agreement"
APPROVED_USAGE = "template_proof_path" if USE_SAMPLE_DATA else "TODO_replace_approved_usage"
ENV_NAME = "dev"
SOURCE_LAYER = "source"
SOURCE_KIND = "lakehouse"
SOURCE_TABLE = "minimal_source" if USE_SAMPLE_DATA else "TODO_source_table"
DATASET_NAME = "sample_agreement_dataset" if USE_SAMPLE_DATA else "topic_dataset"


In [ ]:
CONFIG = load_fabric_config(CONFIG)
validate_notebook_name()
assert_notebook_name_valid()
runtime_context = build_runtime_context(dataset_name=DATASET_NAME, environment=ENV_NAME, source_table=SOURCE_TABLE, target_table="unified.topic_dataset")
if not USE_SAMPLE_DATA:
    bootstrap_fabric_env(config=CONFIG, env=ENV_NAME, required_targets=["source", "unified", "product"])


In [ ]:
source_path = get_path(ENV_NAME, SOURCE_LAYER, config=CONFIG) if not USE_SAMPLE_DATA else None


In [ ]:
if SOURCE_KIND == "lakehouse":
    if USE_SAMPLE_DATA:
        source_path = get_path(ENV_NAME, SOURCE_LAYER, config=CONFIG)
        seed_minimal_sample_source_table(source_path, table_name=SOURCE_TABLE)
    df_source = lakehouse_table_read(source_path, SOURCE_TABLE)
else:
    df_source = warehouse_read(env=ENV_NAME, target=SOURCE_LAYER, schema="dbo", table=SOURCE_TABLE, config=CONFIG)


In [ ]:
print(df_source.schema)
display(df_source.limit(20))


## Segment 2: Profile source and capture evidence

What this does: loads source data and builds profiling metadata for review.

Functions used: `lakehouse_table_read`/`warehouse_read`, `generate_metadata_profile`, `profile_dataframe_to_metadata`.

Output: source profile rows for human review and downstream drafting.


In [ ]:
profile_rows = generate_metadata_profile(df_source, dataset_name=DATASET_NAME)
metadata_rows = profile_dataframe_to_metadata(df_source, dataset_name=DATASET_NAME)
display(profile_rows)
display(metadata_rows)

prompt = suggest_dq_rules_prompt(
    profile_df=profile_rows,
    table_name=SOURCE_TABLE,
    business_context="Describe what this table is used for.",
    output_format="python_config",
)
print(prompt)


## Exploration findings
Document EDA findings here.


## Transformation rationale
Document approved transformation rationale for pipeline handoff.


## Segment 3: AI-assisted drafting (advisory only)

What this does: creates prompt context and candidate DQ/governance suggestions.

Functions used: `build_ai_quality_context`, `build_manual_dq_rule_prompt_package`, `generate_dq_rule_candidates_with_fabric_ai`, `generate_governance_candidates_with_fabric_ai`.

Output: candidate suggestions that require human approval.


In [ ]:
print("AI advisory only; human approval is required before any pipeline enforcement.")


## Draft source input contract

This notebook helps humans define the **source input contract** for the pipeline.

- Capture required columns, optional columns, business keys, classifications, and approved DQ rules.
- In Fabric-first usage, draft this as a Python dict or small table, then write approved values to metadata tables.
- AI suggestions are advisory only and require human/steward approval.
- Optional export formats can be added in project implementations.


In [ ]:
HUMAN_APPROVED_RULES = [
    {"rule_id": "r_customer_not_null", "rule_type": "not_null", "columns": ["customer_id"], "severity": "warning", "description": "customer_id is required"},
    {"rule_id": "r_customer_unique", "rule_type": "unique_key", "columns": ["customer_id"], "severity": "warning", "description": "customer_id should be unique"},
    {"rule_id": "r_event_ts_not_null", "rule_type": "not_null", "columns": ["event_ts"], "severity": "warning", "description": "event_ts is required"},
    {"rule_id": "r_status_allowed", "rule_type": "accepted_values", "columns": ["status"], "severity": "warning", "description": "status within approved values", "allowed_values": ["active", "inactive", "pending"]},
    {"rule_id": "r_amount_non_negative", "rule_type": "value_range", "columns": ["amount"], "severity": "warning", "description": "amount must be non-negative", "min_value": 0},
    {"rule_id": "r_email_regex", "rule_type": "regex_format", "columns": ["email"], "severity": "warning", "description": "email format check", "regex_pattern": r"^[^@\s]+@[^@\s]+\.[^@\s]+$"},
]

SOURCE_INPUT_CONTRACT_APPROVED = normalize_contract_dict({
    "contract_id": f"{DATASET_NAME}_{SOURCE_TABLE}_source_input_v1",
    "contract_type": "source_input",
    "dataset_name": DATASET_NAME,
    "object_name": SOURCE_TABLE,
    "version": "1.0.0",
    "status": "approved",
    "description": "Sample approved source contract built in 02_ex.",
    "grain": "One row per customer event.",
    "required_columns": ["customer_id", "event_ts", "status", "amount", "email", "country_code"],
    "optional_columns": [],
    "business_keys": ["customer_id"],
    "classifications": {},
    "quality_rules": HUMAN_APPROVED_RULES,
    "column_types": {
        "customer_id": {"logical_type": "identifier", "physical_type": "string"},
        "event_ts": {"logical_type": "event_timestamp", "physical_type": "timestamp"},
        "status": {"logical_type": "categorical", "physical_type": "string"},
        "amount": {"logical_type": "measure", "physical_type": "double"},
        "email": {"logical_type": "email", "physical_type": "string"},
        "country_code": {"logical_type": "country_code", "physical_type": "string"},
    },
    "approved_by": "sample_reviewer",
    "approval_note": "Approved in 02_ex sample mode.",
})

display(SOURCE_INPUT_CONTRACT_APPROVED)


## Human-reviewed DQ decisions
Copy only approved deterministic rules to 03_pc notebook.


In [ ]:
print("Optional: governance helpers can be added later; minimal sample focuses on contract + DQ handoff.")


## Segment 4: Human approval and contract write

What this does: captures approved classifications/rules and writes approved contract metadata.

Functions used: `classify_columns`, `summarize_governance_classifications`, `validate_contract_dict`, `write_contract_to_lakehouse`.

Output: approved contract and governance evidence for pipeline enforcement.


## Human-reviewed classification decisions
Approve labels with governance stewards.


In [ ]:
# Optional governance evidence omitted for minimal path.


## Approve and write source input contract
Validate draft contracts, then explicitly gate writes to the dedicated metadata target.


In [ ]:
contract_errors = validate_contract_dict(SOURCE_INPUT_CONTRACT_APPROVED)
if contract_errors:
    raise ValueError("Contract validation failed: " + " | ".join(contract_errors))

WRITE_APPROVED_CONTRACT = True
USE_LOCAL_SAMPLE_METADATA = False
# Set True only when running locally without Fabric metadata tables.

if WRITE_APPROVED_CONTRACT and not USE_LOCAL_SAMPLE_METADATA:
    metadata_path = get_path(ENV_NAME, "metadata", config=CONFIG)
    write_contract_to_lakehouse(SOURCE_INPUT_CONTRACT_APPROVED, metadata_path)
elif USE_LOCAL_SAMPLE_METADATA:
    out = Path("samples/end_to_end/_output/metadata")
    out.mkdir(parents=True, exist_ok=True)
    (out / "contracts.json").write_text(json.dumps([SOURCE_INPUT_CONTRACT_APPROVED], default=str, indent=2), encoding="utf-8")
    print("Wrote local sample metadata artifact: samples/end_to_end/_output/metadata/contracts.json")


In [ ]:
from fabricops_kit.data_lineage import build_lineage_from_notebook_code

# Optional documentation helper only (not transformation logic).
# Paste or load exported notebook source when ready to document lineage.
# result = build_lineage_from_notebook_code(notebook_code)
# display(result["deterministic_steps"])
# print(result["copilot_prompt"])
